In [ ]:
import os
os.chdir("../")

import torch

device = torch.device(f'cuda:0')
dtype = torch.float32

# Image

## K-VAE-2d

In [ ]:
from kvae.models import KVAEImage

model_paths = {
        "KVAE_1.0": "kandinskylab/KVAE-2D-1.0",
        "KVAE_2.0": "kandinskylab/KVAE-2D-2.0",
    }

model = 'KVAE_2.0'

vae = KVAEImage.from_pretrained(model_paths[model]).eval().to(device).to(dtype)

## Metrics

In [ ]:
from torchmetrics import MetricCollection
from torchmetrics.image import  PeakSignalNoiseRatio, LearnedPerceptualImagePatchSimilarity

metrics = MetricCollection(
    {
        "psnr": PeakSignalNoiseRatio(data_range=(-1, 1), reduction="none", dim=[1, 2, 3]),
        "lpips": LearnedPerceptualImagePatchSimilarity(reduction="none", net_type="alex"),
    }
).to(device=device)

## Prepare image

In [ ]:
from pathlib import Path
from data import read_image, _norm_to_255numpy, save_tensor_image
import mediapy

image_path = Path('/home/jovyan/kobenko/PROD/kvae-1/assets/images/0004.png')
image = read_image(image_path).unsqueeze(0).to(dtype=dtype, device=device)

with torch.no_grad():
    latent = vae.encode(image).latent_dist.mode()
    recon = vae.decode(latent).clip(-1, 1)

    # Compute Metrics
    result = metrics(recon.float(), image.float())

print(f"PSNR (dB): {result['psnr'].item():.4f}")
print(f"LPIPS    : {result['lpips'].item():.4f}")

# Show image, reconstruction
show_image = _norm_to_255numpy(image.squeeze(0).float().cpu().permute(1,2,0), input_norm='-11')
show_recon = _norm_to_255numpy(recon.squeeze(0).float().cpu().permute(1,2,0), input_norm='-11')
show_diff = _norm_to_255numpy((abs(recon-image)-1).squeeze(0).float().cpu().permute(1,2,0), input_norm='-11')
mediapy.show_images([show_image, show_recon, show_diff], ["Input Image", "Reconstructed Image", "Difference"], height=720)

# Saving reconstruction
saving_folder = Path('./outputs/images')
save_tensor_image(
    recon.float().cpu().squeeze(0),
    save_dir_path=saving_folder,
    filename=image_path.name,
)

# Video

## K-VAE-3d

In [ ]:
from kvae.models import KVAE3D

model_paths = {
        "KVAE_1.0": "kandinskylab/KVAE-3D-1.0",
        "KVAE_2.0-t4s8": "kandinskylab/KVAE-3D-2.0-t4s8",
        "KVAE_2.0-t4s16": "kandinskylab/KVAE-3D-2.0-t4s16",
    }

model = 'KVAE_1.0'

vae = (
    KVAE3D.from_pretrained(model_paths[model]).eval().to(device).to(dtype)
)

## Metrics

In [ ]:
from torchmetrics import MetricCollection
from metrics.video_metrics import VideoPSNR, VideoLPIPS

metrics = MetricCollection(
    {
        "psnr": VideoPSNR(data_range=(-1, 1), metric_chank_size=10),
        "lpips": VideoLPIPS(net_type="alex", metric_chank_size=10),
    }
).to(device=device)

## Prepare video

In [ ]:
from pathlib import Path
from data import VideoReader, _norm_to_255numpy, quant_renormalization
import mediapy

video_path = Path('./assets/video_test/31')

reader = VideoReader(stream_pattern='*.png', input_norm='m11')
video = reader.read_video(video_path)['frames'].unsqueeze(0).to(dtype=dtype, device=device)

with torch.no_grad():
    latent = vae.encode(video).latent_dist.mode()
    recon = vae.decode(latent).clip(-1, 1)

    # Compute Metrics
    video = quant_renormalization(video, input_norm='m11', output_norm='-11').squeeze(0).float()
    recon = quant_renormalization(recon, input_norm='m11', output_norm='-11').squeeze(0).float()
    result = metrics(recon, video)

print(f"PSNR (dB): {result['psnr_metric_per_video'].item():.4f}")
print(f"LPIPS    : {result['lpips_metric_per_video'].item():.4f}")

# Show video, reconstruction
show_video = _norm_to_255numpy(video.cpu().permute(0,2,3,1), input_norm='-11')
show_recon = _norm_to_255numpy(recon.cpu().permute(0,2,3,1), input_norm='-11')
mediapy.show_videos([show_video, show_recon], ["Input Image", "Reconstructed Image"], fps=24)

# Saving reconstruction
saving_folder = Path('./outputs/videos')
output_filepath = saving_folder / (video_path.stem + '.mp4')
mediapy.write_video(output_filepath, show_recon, fps=24)
print(f'Saving path for video result:\t{str(output_filepath)}')

# Audio

## K-VAE-1d

In [ ]:
from kvae.models import KVAEAudio

# audio_model_paths = {"KVAE-Audio": "kandinskylab/KVAE-Audio"}
audio_model_paths = {"KVAE-Audio": "/home/jovyan/kobenko/PROD/release_checkpoints/KVAE-Audio"}
audio_model_name = "KVAE-Audio"

# Keep float32 for parity with the original KVAE-Audio inference.
audio_dtype = torch.float32
vae = KVAEAudio.from_pretrained(audio_model_paths[audio_model_name]).eval().to(device=device, dtype=audio_dtype)


## Metrics

In [ ]:
from metrics.audio_metrics import (
    MelSpectrogramDistance,
    MultiScaleSTFTDistance,
    SISDRDistance,
    WaveformL1Distance,
)

audio_distances = {
    "waveform_l1": WaveformL1Distance().to(device),
    "stft": MultiScaleSTFTDistance().to(device),
    "mel": MelSpectrogramDistance().to(device),
    "sisdr": SISDRDistance().to(device),
}


## Prepare audio

In [ ]:
from pathlib import Path

from audiotools import AudioSignal
from IPython.display import Audio, display

from data import read_audio, save_audio

audio_root_candidates = (Path.cwd(), Path.cwd() / "kvae-1", Path.cwd().parent)
audio_project_root = next(
    (root for root in audio_root_candidates if (root / "assets/audio_test").exists()),
    None,
)
if audio_project_root is None:
    raise FileNotFoundError(
        "Could not find the project root containing assets/audio_test"
    )

audio_path = audio_project_root / "assets/audio_test/98537484.wav"
input_signal = read_audio(audio_path).to(device)
audio = input_signal.audio_data.to(dtype=audio_dtype)
sample_rate = input_signal.sample_rate

with torch.no_grad():
    posterior = vae.encode(audio, sample_rate).latent_dist
    latent = posterior.mode()
    reconstruction = vae.decode(latent)[..., : audio.shape[-1]]

reference_signal = AudioSignal(audio.float(), sample_rate)
reconstruction_signal = AudioSignal(reconstruction.float(), sample_rate)
metric_values = {
    "waveform_l1": audio_distances["waveform_l1"](
        reference_signal, reconstruction_signal
    ).item(),
    "stft": audio_distances["stft"](
        reference_signal, reconstruction_signal
    ).item(),
    "mel": audio_distances["mel"](
        reference_signal, reconstruction_signal
    ).item(),
    "sisdr_db": -audio_distances["sisdr"](
        reference_signal, reconstruction_signal
    ).item(),
}

audio_saving_folder = audio_project_root / "outputs/audio"
saved_audio_path = save_audio(
    tensor=reconstruction.float(),
    reference_tensor=audio.float(),
    sample_rate=sample_rate,
    save_dir_path=audio_saving_folder,
    filename=audio_path.name,
)
saved_reconstruction = read_audio(saved_audio_path)

print("Input shape:", tuple(audio.shape))
print("Latent shape:", tuple(latent.shape))
print("Reconstruction shape:", tuple(reconstruction.shape))
print("Sample rate:", sample_rate)
print("Metrics:", metric_values)
print("Saved to:", saved_audio_path)

print("Original")
display(Audio(input_signal.audio_data[0, 0].detach().cpu().numpy(), rate=sample_rate))
print("Saved reconstruction")
display(Audio(saved_reconstruction.audio_data[0, 0].numpy(), rate=sample_rate))
